# 03 Researcher Agent

In [1]:
# Start coding here
from typing import TypedDict
from typing import List
from typing import Dict

In [2]:
class ResearchState(TypedDict):

    research_goal: str

    queries_executed: List[str]

    raw_findings: List[str]

    deduplicated_findings: List[str]

    conflicts: List[str]

    critic_score: float

    draft_report: str

    final_report: str

    iteration_count: int

    status: str

    errors: List[str]

    metadata: Dict

In [4]:
from langgraph.graph import StateGraph, END

from transformers import pipeline

In [5]:
llm = pipeline(
    "text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", max_new_tokens=256
)

Device set to use cpu


In [6]:
def search_tool(query):

    mock_database = {
        "cloud costs": [
            "AWS pricing increased 8%",
            "Azure pricing remained stable",
            "Google Cloud storage reduced 4%",
        ],
        "competitors": ["Oracle Cloud increased 3%", "IBM Cloud remained unchanged"],
    }

    results = []

    for key, value in mock_database.items():

        if key.lower() in query.lower():

            results.extend(value)

    return results

In [7]:
search_tool("cloud costs")

['AWS pricing increased 8%',
 'Azure pricing remained stable',
 'Google Cloud storage reduced 4%']

In [8]:
def generate_research_queries(goal):

    prompt = f"""
Generate 3 research queries.

Goal:

{goal}
"""

    result = llm(prompt)

    text = result[0]["generated_text"]

    return text

In [9]:
def generate_queries(goal):

    return [f"{goal} AWS", f"{goal} Azure", f"{goal} Google Cloud"]

In [10]:
generate_queries("Compare cloud costs")

['Compare cloud costs AWS',
 'Compare cloud costs Azure',
 'Compare cloud costs Google Cloud']

In [11]:
def researcher_agent(state: ResearchState):

    print("\nResearcher Agent Running...")

    goal = state["research_goal"]

    queries = generate_queries(goal)

    findings = []

    for query in queries:

        state["queries_executed"].append(query)

        results = search_tool(query)

        findings.extend(results)

    state["raw_findings"].extend(findings)

    state["status"] = "research_completed"

    return state

In [12]:
initial_state = {
    "research_goal": "cloud costs",
    "queries_executed": [],
    "raw_findings": [],
    "deduplicated_findings": [],
    "conflicts": [],
    "critic_score": 0.0,
    "draft_report": "",
    "final_report": "",
    "iteration_count": 0,
    "status": "",
    "errors": [],
    "metadata": {},
}

In [13]:
updated_state = researcher_agent(initial_state)


Researcher Agent Running...


In [14]:
updated_state["queries_executed"]

['cloud costs AWS', 'cloud costs Azure', 'cloud costs Google Cloud']

In [15]:
updated_state["raw_findings"]

['AWS pricing increased 8%',
 'Azure pricing remained stable',
 'Google Cloud storage reduced 4%',
 'AWS pricing increased 8%',
 'Azure pricing remained stable',
 'Google Cloud storage reduced 4%',
 'AWS pricing increased 8%',
 'Azure pricing remained stable',
 'Google Cloud storage reduced 4%']

In [16]:
from datetime import datetime

In [17]:
def log_query(state, query):

    if "query_log" not in state["metadata"]:

        state["metadata"]["query_log"] = []

    state["metadata"]["query_log"].append(
        {"query": query, "timestamp": str(datetime.now())}
    )

In [18]:
def enhanced_researcher_agent(state):

    goal = state["research_goal"]

    queries = generate_queries(goal)

    findings = []

    for query in queries:

        log_query(state, query)

        state["queries_executed"].append(query)

        findings.extend(search_tool(query))

    state["raw_findings"].extend(findings)

    return state

In [19]:
state = enhanced_researcher_agent(initial_state)

state["metadata"]

{'query_log': [{'query': 'cloud costs AWS',
   'timestamp': '2026-06-19 14:45:44.882108'},
  {'query': 'cloud costs Azure', 'timestamp': '2026-06-19 14:45:44.882108'},
  {'query': 'cloud costs Google Cloud',
   'timestamp': '2026-06-19 14:45:44.882108'}]}

In [20]:
graph = StateGraph(ResearchState)

In [21]:
graph.add_node("researcher", researcher_agent)

In [22]:
graph.set_entry_point("researcher")

graph.add_edge("researcher", END)

In [23]:
app = graph.compile()

In [24]:
result = app.invoke(initial_state)

result


Researcher Agent Running...


{'research_goal': 'cloud costs',
 'queries_executed': ['cloud costs AWS',
  'cloud costs Azure',
  'cloud costs Google Cloud',
  'cloud costs AWS',
  'cloud costs Azure',
  'cloud costs Google Cloud',
  'cloud costs AWS',
  'cloud costs Azure',
  'cloud costs Google Cloud'],
 'raw_findings': ['AWS pricing increased 8%',
  'Azure pricing remained stable',
  'Google Cloud storage reduced 4%',
  'AWS pricing increased 8%',
  'Azure pricing remained stable',
  'Google Cloud storage reduced 4%',
  'AWS pricing increased 8%',
  'Azure pricing remained stable',
  'Google Cloud storage reduced 4%',
  'AWS pricing increased 8%',
  'Azure pricing remained stable',
  'Google Cloud storage reduced 4%',
  'AWS pricing increased 8%',
  'Azure pricing remained stable',
  'Google Cloud storage reduced 4%',
  'AWS pricing increased 8%',
  'Azure pricing remained stable',
  'Google Cloud storage reduced 4%',
  'AWS pricing increased 8%',
  'Azure pricing remained stable',
  'Google Cloud storage reduced